# Lab 5 – Evaluation: RAGAS + LLM-as-judge + tracing

Đánh giá câu trả lời của Lab 4 (prompt `basic` v2) trên 20 câu trong phạm vi:
1. **Tracing**: chạy lại pipeline có trace từng bước (LangFuse nếu đang chạy, nếu không ghi JSONL cục bộ).
2. **RAGAS** (judge `gpt-4.1-mini`, embedding `text-embedding-3-small`): faithfulness, answer relevancy, context precision, context recall, factual correctness.
3. **LLM-as-judge** tự viết (rubric 4 nhãn) và **đối chiếu với nhãn người** (đọc tay ở Lab 4).
4. **Chẩn đoán**: lỗi retrieval hay lỗi generation cho từng câu.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 230)

from ragbv.config import load_manifest, REPORTS, DATA
from ragbv import chunking as C, store, retrieval as R, generation as G, evaluation as EV
from ragbv.tracing import Tracer, langfuse_available
from ragbv.golden import load_golden, is_relevant

golden = load_golden()
runs = {json.loads(l)["id"]: json.loads(l) for l in (DATA / "runs" / "lab4_basic.jsonl").read_text(encoding="utf-8").splitlines()}
print("LangFuse đang chạy:", langfuse_available())

LangFuse đang chạy: False


## 1. Chạy pipeline có trace

In [2]:
docs = load_manifest(); chunks = C.load_chunks("structure"); conn = store.connect()
ret = R.Retriever(chunks, docs, conn, "structure")
RETRIEVE = dict(method="hybrid3+rerank", k=5, n_cand=20, use_filter=True, use_rewrite=True)
tracer = Tracer("lab5_pipeline")
print("backend:", tracer.backend, "->", tracer.path)
for g in golden:
    with tracer.trace("rag_answer", {"question": g.question}, {"golden_id": g.id, "type": g.type}) as t:
        timings = {}
        with t.span("retrieve", {"config": RETRIEVE}) as s:
            ctx = ret.run(g.question, timings=timings, **RETRIEVE)
            s["output"] = {"chunk_ids": [c.chunk_id for c in ctx], "docs": [c.doc_id for c in ctx],
                           "rewritten": R.rewrite(g.question), "filter": R.product_filter(g.question, docs),
                           "step_ms": {k: round(v * 1000, 1) for k, v in timings.items()}}
        with t.span("generate", {"prompt": "basic"}) as s:
            out = G.generate(g.question, ctx, "basic")
            s["output"] = {"status": out["status"], "usage": out["usage"], "llm_latency_s": out["latency_s"]}
        with t.span("check_citations") as s:
            s["output"] = G.check(out, ctx)
        t.output = {"status": out["status"], "answer": out["answer"]}
        t.score("cites_valid", float(not s["output"]["invalid_ids"]))
        assert [c.chunk_id for c in ctx] == runs[g.id]["context_ids"], f"{g.id}: ngữ cảnh khác Lab 4"
tracer.flush()
traces = [json.loads(l) for l in tracer.path.read_text(encoding="utf-8").splitlines()]
print(len(traces), "trace")
print(json.dumps(traces[0], ensure_ascii=False, indent=1)[:1500])

backend: local -> C:\Users\Legion\Desktop\Job\Paraline\Lab\data\traces\lab5_pipeline.jsonl


C:\Users\Legion\Desktop\Job\Paraline\Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   5%|▌         | 21/393 [00:00<00:02, 170.78it/s]

Loading weights:  10%|▉         | 39/393 [00:00<00:03, 117.64it/s]

Loading weights:  13%|█▎        | 52/393 [00:00<00:03, 94.20it/s] 

... (6 outputs left out)

Loading weights:  30%|███       | 119/393 [00:01<00:04, 55.83it/s]

Loading weights:  32%|███▏      | 126/393 [00:01<00:05, 51.89it/s]

Loading weights: 100%|██████████| 393/393 [00:01<00:00, 217.68it/s]

25 trace
{
 "trace_id": "77cd48767ac94d5aa60b016a656bb6ef",
 "run": "lab5_pipeline",
 "name": "rag_answer",
 "time": "2026-10-21T10:40:39.390486+00:00",
 "input": {
  "question": "Phí bảo hiểm điều khoản chính (điều trị nội trú) gói Bạc của Bảo Việt An Gia cho người 35 tuổi là bao nhiêu một năm?"
 },
 "output": {
  "status": "tra_loi",
  "answer": "Phí bảo hiểm điều khoản chính (điều trị nội trú) gói Bạc của Bảo Việt An Gia cho người 35 tuổi là 1.887.600 đồng một năm [C3]."
 },
 "metadata": {
  "golden_id": "G01",
  "type": "table_lookup"
 },
 "duration_ms": 37251.2,
 "spans": [
  {
   "name": "retrieve",
   "input": {
    "config": {
     "method": "hybrid3+rerank",
     "k": 5,
     "n_cand": 20,
     "use_filter": true,
     "use_rewrite": true
    }
   },
   "output": {
    "chunk_ids": [
     "c3d95d23a428",
     "67e60a5b07f1",
     "97944fd21cc7",
     "f3fdad2b1c2b",
     "9befd1cda530"
    ],
    "docs": [
     "bv-angia-2016",
     "bv-angia-2016",
     "bv-angia-2016",
     

In [3]:
# Thời gian từng bước, đọc từ trace (rewrite/generation lấy từ cache nên thời gian ở đây là thời gian chạy lại;
# thời gian gọi LLM thật lấy từ llm_latency_s đã lưu lúc sinh ở Lab 4)
steps = []
for tr in traces:
    sp = {s["name"]: s for s in tr["spans"]}
    step_ms = sp["retrieve"]["output"]["step_ms"]
    steps.append({"id": tr["metadata"]["golden_id"], **{f"retr_{k}": v for k, v in step_ms.items()},
                  "llm_generate_ms": sp["generate"]["output"]["llm_latency_s"] * 1000,
                  "prompt_tokens": sp["generate"]["output"]["usage"]["prompt_tokens"],
                  "completion_tokens": sp["generate"]["output"]["usage"]["completion_tokens"]})
steps = pd.DataFrame(steps)
steps.describe(percentiles=[0.5, 0.95]).loc[["50%", "95%", "mean"]].round(1)

,retr_rewrite,retr_dense,retr_bm25,retr_bm25_nd,retr_rerank,llm_generate_ms,prompt_tokens,completion_tokens
50%,1.1,11.7,3.1,3.1,1008.6,5030.0,2371.0,170.0
95%,2.4,31.7,6.3,4.7,1111.9,6598.0,2911.6,351.8
mean,1.3,15.3,3.9,3.2,2443.7,5191.6,2434.6,178.5


## 2. RAGAS (20 câu trong phạm vi)

In [4]:
ins = [g for g in golden if g.in_scope]
rows = [{"question": g.question, "answer": runs[g.id]["answer"], "contexts": runs[g.id]["contexts"], "reference": g.answer} for g in ins]
rg = EV.ragas_eval(rows)
rg.insert(0, "id", [g.id for g in ins]); rg.insert(1, "type", [g.type for g in ins])
mcols = [c for c in EV.RAGAS_METRICS if c in rg.columns]
print(rg[mcols].mean().round(3))
rg[["id", "type", *mcols]].round(2)

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


... (8 outputs left out)

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


faithfulness                            0.871
answer_relevancy                        0.612
llm_context_precision_with_reference    0.793
context_recall                          0.888
factual_correctness(mode=f1)            0.453
dtype: float64


,id,type,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall,factual_correctness(mode=f1)
0,G01,table_lookup,1.00,0.99,0.50,1.00,0.67
1,G02,table_lookup,1.00,0.68,1.00,1.00,0.00
2,G03,table_lookup,0.75,0.63,0.42,0.50,0.33
3,G04,multi_condition,1.00,0.67,1.00,0.67,0.73
4,G05,fact,1.00,0.60,1.00,1.00,0.80
5,G06,fact,1.00,1.00,1.00,1.00,0.44
6,G07,fact,1.00,0.28,1.00,1.00,0.60
7,G08,table_lookup,1.00,0.64,1.00,1.00,0.67
8,G09,multi_condition,1.00,0.59,1.00,1.00,0.89
9,G10,fact,1.00,0.19,0.50,1.00,0.40


## 3. LLM-as-judge và đối chiếu nhãn người
Nhãn người: đọc tay 20 câu trả lời ở Lab 4 (`reports/lab4.md`).

In [5]:
HUMAN = {**{i: "dung_du" for i in ["G01", "G02", "G04", "G05", "G06", "G07", "G08", "G09", "G10", "G11", "G13", "G14", "G15", "G20"]},
         **{i: "thieu_y" for i in ["G12", "G16", "G17", "G18", "G03"]}, "G19": "sai"}
jd = pd.DataFrame([{"id": g.id, "human": HUMAN[g.id], **EV.judge(g.question, g.answer, runs[g.id]["answer"])} for g in ins])
agree = (jd.human == jd.label).mean()
print(f"Đồng thuận judge–người: {agree:.0%} ({(jd.human == jd.label).sum()}/{len(jd)})")
print(pd.crosstab(jd.human, jd.label, rownames=["người"], colnames=["judge"]))
jd[jd.human != jd.label]

Đồng thuận judge–người: 70% (14/20)
judge    dung_du  thieu_y
người                    
dung_du        9        5
sai            0        1
thieu_y        0        5


,id,human,label,reason
3,G04,dung_du,thieu_y,"Chatbot trả lời đầy đủ về độ tuổi, điều kiện tham gia và thời gian chờ 635 ngày nhưng ..."
5,G06,dung_du,thieu_y,"Chatbot chỉ nêu thời gian chờ 01 năm cho bệnh có sẵn và bệnh đặc biệt, thiếu các thời ..."
9,G10,dung_du,thieu_y,Chatbot trả lời đúng về việc hoàn trả 80% phí bảo hiểm khi người mua yêu cầu hủy và đi...
10,G11,dung_du,thieu_y,Chatbot chỉ nêu thời gian chờ là 270 ngày mà không đề cập đến điều kiện ngoại lệ là cá...
13,G14,dung_du,thieu_y,"Chatbot nêu đúng điều kiện và mức giảm 15% cho 3 năm liên tục, nhưng không đề cập đến ..."
18,G19,sai,thieu_y,Chatbot chỉ nêu tuổi tối đa kết thúc bảo hiểm là 76 tuổi nhưng không đề cập đến thời h...


In [6]:
# RAGAS factual correctness có tách được câu sai/thiếu khỏi câu đúng không?
fc = rg.set_index("id")["factual_correctness(mode=f1)"]
cmp = pd.DataFrame({"human": pd.Series(HUMAN), "factual_correctness": fc, "faithfulness": rg.set_index("id")["faithfulness"]})
cmp.groupby("human").agg(["mean", "min", "count"]).round(2)

factual_correctness             faithfulness            
                       mean   min count         mean   min count
human                                                           
dung_du                0.49  0.00    14         1.00  1.00    14
sai                    0.29  0.29     1         0.67  0.67     1
thieu_y                0.38  0.00     5         0.55  0.00     5

## 4. Chẩn đoán từng câu: retrieval hay generation?
Quy tắc: context recall thấp (< 0.5) → lỗi **retrieval**; recall đủ nhưng faithfulness thấp (< 0.7) → generation **không bám ngữ cảnh**; recall và faithfulness đủ nhưng nhãn người là thiếu/sai → generation **thiếu ý hoặc diễn giải sai**.

In [7]:
lab4 = {r["id"]: r for r in json.loads((REPORTS / "lab4_results.json").read_text(encoding="utf-8"))["per_question"] if r["prompt"] == "basic"}
diag = rg.set_index("id")[mcols].copy()
diag["gold_in_ctx"] = [lab4[i]["gold_in_context"] for i in diag.index]
diag["human"] = [HUMAN[i] for i in diag.index]
diag["judge"] = jd.set_index("id").label
def diagnose(r):
    if r["context_recall"] < 0.5: return "retrieval"
    if r["faithfulness"] < 0.7: return "generation: không bám ngữ cảnh"
    if r["human"] != "dung_du": return "generation: thiếu ý / diễn giải sai"
    return "ok"
diag["chan_doan"] = diag.apply(diagnose, axis=1)
print(diag.chan_doan.value_counts())
diag.sort_values(["chan_doan", "context_recall"]).round(2)

chan_doan
ok                                     14
generation: thiếu ý / diễn giải sai     3
generation: không bám ngữ cảnh          2
retrieval                               1
Name: count, dtype: int64


,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall,factual_correctness(mode=f1),gold_in_ctx,human,judge,chan_doan
id,,,,,,,,,
G17,0.00,0.64,0.00,0.50,0.75,True,thieu_y,thieu_y,generation: không bám ngữ cảnh
G19,0.67,0.34,1.00,1.00,0.29,True,sai,thieu_y,generation: không bám ngữ cảnh
G03,0.75,0.63,0.42,0.50,0.33,False,thieu_y,thieu_y,generation: thiếu ý / diễn giải sai
G12,1.00,0.45,0.83,0.75,0.00,True,thieu_y,thieu_y,generation: thiếu ý / diễn giải sai
G18,1.00,0.79,1.00,1.00,0.25,True,thieu_y,thieu_y,generation: thiếu ý / diễn giải sai
G04,1.00,0.67,1.00,0.67,0.73,True,dung_du,thieu_y,ok
G01,1.00,0.99,0.50,1.00,0.67,True,dung_du,dung_du,ok
G02,1.00,0.68,1.00,1.00,0.00,True,dung_du,dung_du,ok
G05,1.00,0.60,1.00,1.00,0.80,True,dung_du,dung_du,ok


## 5. Lưu

In [8]:
out = {"langfuse_running": langfuse_available(), "trace_backend": tracer.backend, "trace_file": str(tracer.path),
       "step_latency": steps.describe(percentiles=[0.5, 0.95]).round(1).to_dict(),
       "ragas_mean": rg[mcols].mean().round(3).to_dict(), "ragas": rg[["id", "type", *mcols]].round(3).to_dict("records"),
       "judge": jd.to_dict("records"), "judge_agreement": agree,
       "judge_confusion": pd.crosstab(jd.human, jd.label).to_dict(),
       "diagnosis": diag.reset_index().round(3).to_dict("records")}
(REPORTS / "lab5_results.json").write_text(json.dumps(out, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("saved")

saved
